### Proyecto 4: traductor Python -> C++, medir ganancia real

El LLM traduce una funcion Python a C++ equivalente. Compilamos el C++ con `g++ -O3` y cronometramos las dos versiones con el mismo input. La ganancia de velocidad no se "intuye" mirando el codigo -- se mide.

Funcion elegida: contar numeros primos hasta N por division de prueba (fuerza bruta, intencionalmente ineficiente en Python puro para que el contraste con C++ compilado sea grande).

In [1]:
import time

N = 200_000

def contar_primos(n):
    def es_primo(x):
        if x < 2:
            return False
        for d in range(2, int(x ** 0.5) + 1):
            if x % d == 0:
                return False
        return True
    return sum(1 for x in range(2, n) if es_primo(x))

inicio = time.perf_counter()
resultado_python = contar_primos(N)
tiempo_python = time.perf_counter() - inicio

print(f"Python: {resultado_python} primos en {tiempo_python:.3f}s")

Python: 17984 primos en 0.163s


#### Traducir a C++ con Gemini

Le pedimos el equivalente en C++, mismo algoritmo (division de prueba, no le pedimos que use una criba mas inteligente -- queremos medir la ganancia de compilar/tipar, no de cambiar de algoritmo). Le exigimos que imprima el resultado como unica salida para poder comparar.

In [2]:
from dotenv import load_dotenv
from google import genai
import os

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

prompt = f"""Traduci esta funcion Python a C++ (mismo algoritmo, division de prueba, sin optimizar el algoritmo -- solo traducir).

def contar_primos(n):
    def es_primo(x):
        if x < 2:
            return False
        for d in range(2, int(x ** 0.5) + 1):
            if x % d == 0:
                return False
        return True
    return sum(1 for x in range(2, n) if es_primo(x))

print(contar_primos({N}))

Devolveme SOLO el codigo C++ completo (con #include, main, todo), sin explicacion, sin markdown, sin ```. El programa debe imprimir unicamente el numero resultado, nada mas."""

response = client.models.generate_content(
    model="gemini-flash-lite-latest",
    contents=prompt,
)
codigo_cpp = response.text.strip()
print(codigo_cpp)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


#include <iostream>
#include <cmath>

bool es_primo(int x) {
    if (x < 2) {
        return false;
    }
    int limite = static_cast<int>(std::sqrt(x));
    for (int d = 2; d <= limite; ++d) {
        if (x % d == 0) {
            return false;
        }
    }
    return true;
}

int contar_primos(int n) {
    int suma = 0;
    for (int x = 2; x < n; ++x) {
        if (es_primo(x)) {
            suma += 1;
        }
    }
    return suma;
}

int main() {
    std::cout << contar_primos(200000) << std::endl;
    return 0;
}


#### Compilar y correr

`-O3` activa las optimizaciones del compilador -- sin esto la comparacion no seria justa (C++ sin optimizar tambien puede ser lento).

In [3]:
import subprocess

with open("primos.cpp", "w") as f:
    f.write(codigo_cpp)

compilacion = subprocess.run(
    ["g++", "-O3", "-o", "primos", "primos.cpp"],
    capture_output=True, text=True,
)
assert compilacion.returncode == 0, f"Error de compilacion:\n{compilacion.stderr}"
print("Compilado ok")

inicio = time.perf_counter()
ejecucion = subprocess.run(["./primos"], capture_output=True, text=True)
tiempo_cpp = time.perf_counter() - inicio
resultado_cpp = int(ejecucion.stdout.strip())

print(f"C++: {resultado_cpp} primos en {tiempo_cpp:.3f}s")

Compilado ok
C++: 17984 primos en 0.011s


In [4]:
assert resultado_python == resultado_cpp, "Los resultados no coinciden -- el C++ generado tiene un bug"

print(f"Python: {tiempo_python:.3f}s")
print(f"C++:    {tiempo_cpp:.3f}s")
print(f"Ganancia real: {tiempo_python / tiempo_cpp:.1f}x mas rapido")

Python: 0.163s
C++:    0.011s
Ganancia real: 14.6x mas rapido
